## 1. Install Dependencies

We install the required libraries (including `tabulate` for Markdown exports) in a dedicated cell.

In [1]:
# Install required libraries
%pip install pandas numpy openpyxl tabulate

Note: you may need to restart the kernel to use updated packages.


## 2. Import Libraries

We import libraries required for generating our data dictionary.

In [2]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

## 3. Project Paths

We dynamically determine the project root folder to ensure the script can find our raw data and save reports correctly.

In [3]:
current_dir = Path.cwd()

# Dynamically resolve the project root
if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

RAW_DATA = PROJECT_ROOT / "data" / "raw"
REPORTS = PROJECT_ROOT / "reports"
DOCS = PROJECT_ROOT / "docs"

# Ensure the directories exist
REPORTS.mkdir(exist_ok=True)
DOCS.mkdir(exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Raw Data:", RAW_DATA)

Project Root: c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL INTELLIGENCE PLATFORM\nifty100-financial-intelligence-platform
Raw Data: c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL INTELLIGENCE PLATFORM\nifty100-financial-intelligence-platform\data\raw


## 4. Find All Excel Files

We locate and list all the Excel datasets in our raw data directory that we will profile for our dictionary.

In [4]:

excel_files = sorted(RAW_DATA.glob("*.xlsx"))

print(f"Datasets Found: {len(excel_files)}")

for file in excel_files:
    print(file.name)

Datasets Found: 7
analysis.xlsx
balancesheet.xlsx
cashflow.xlsx
companies.xlsx
documents.xlsx
profitandloss.xlsx
prosandcons.xlsx


## 5. Generate Dictionary

We iterate through every column in every dataset, calculating data types, missing values, unique values, and extracting a sample value for context.

In [5]:
dictionary = []

for file in excel_files:
    try:
        df = pd.read_excel(file)

        for column in df.columns:
            # Safely grab a sample value if one exists
            valid_data = df[column].dropna()
            sample_val = str(valid_data.iloc[0]) if len(valid_data) > 0 else ""

            dictionary.append({
                "Dataset": file.name,
                "Column Name": column,
                "Data Type": str(df[column].dtype),
                "Missing Values": int(df[column].isnull().sum()),
                "Missing %": round(df[column].isnull().mean()*100, 2),
                "Unique Values": int(df[column].nunique()),
                "Sample Value": sample_val
            })
    except Exception as e:
        print(f"Error reading {file.name}: {e}")

dictionary_df = pd.DataFrame(dictionary)
dictionary_df.head()

Error reading balancesheet.xlsx: [Errno 13] Permission denied: 'c:\\Users\\polai\\OneDrive\\Desktop\\N100 FINANCIAL INTELLIGENCE PLATFORM\\nifty100-financial-intelligence-platform\\data\\raw\\balancesheet.xlsx'


,Dataset,Column Name,Data Type,Missing Values,Missing %,Unique Values,Sample Value
0,analysis.xlsx,Bluestock Fintech — Nifty 100 | Analysis | ...,str,0,0.0,21,id
1,analysis.xlsx,Unnamed: 1,str,0,0.0,6,company_id
2,analysis.xlsx,Unnamed: 2,str,0,0.0,21,compounded_sales_growth
3,analysis.xlsx,Unnamed: 3,str,0,0.0,19,compounded_profit_growth
4,analysis.xlsx,Unnamed: 4,str,0,0.0,21,stock_price_cagr


## 6. Sort Dictionary

We sort our completed dictionary alphabetically by Dataset name and then by Column name to keep it organized.

In [6]:
if not dictionary_df.empty:
    dictionary_df = dictionary_df.sort_values(["Dataset", "Column Name"]).reset_index(drop=True)

dictionary_df

,Dataset,Column Name,Data Type,Missing Values,Missing %,Unique Values,Sample Value
0,analysis.xlsx,Bluestock Fintech — Nifty 100 | Analysis | ...,str,0,0.00,21,id
1,analysis.xlsx,Unnamed: 1,str,0,0.00,6,company_id
2,analysis.xlsx,Unnamed: 2,str,0,0.00,21,compounded_sales_growth
3,analysis.xlsx,Unnamed: 3,str,0,0.00,19,compounded_profit_growth
4,analysis.xlsx,Unnamed: 4,str,0,0.00,21,stock_price_cagr
5,analysis.xlsx,Unnamed: 5,str,0,0.00,21,roe
6,cashflow.xlsx,Bluestock Fintech — Nifty 100 | Cash Flow |...,object,0,0.00,1188,id
7,cashflow.xlsx,Unnamed: 1,str,0,0.00,101,company_id
8,cashflow.xlsx,Unnamed: 2,str,0,0.00,52,year
9,cashflow.xlsx,Unnamed: 3,object,2,0.17,1107,operating_activity


## 7. Export to Excel

We export the full data dictionary as a standalone Excel file in the reports folder for non-technical stakeholders.

In [7]:
if not dictionary_df.empty:
    dictionary_df.to_excel(
        REPORTS / "data_dictionary.xlsx",
        index=False
    )
    print("Excel Exported Successfully!")
else:
    print("No data to export.")

Excel Exported Successfully!


## 8. Export to Markdown

We also generate a structured Markdown document, converting each dataset's dictionary into a clean markdown table using the 'tabulate' library, and save it in the docs folder.

In [8]:
if not dictionary_df.empty:
    markdown = "# Data Dictionary\n\n"

    for dataset in dictionary_df["Dataset"].unique():
        markdown += f"## {dataset}\n\n"
        
        temp = dictionary_df[dictionary_df["Dataset"] == dataset]
        
        # Use tabulate via pandas to_markdown
        markdown += temp.to_markdown(index=False)
        markdown += "\n\n"

    (DOCS / "data_dictionary.md").write_text(
        markdown,
        encoding="utf-8"
    )

    print("Markdown Exported Successfully!")
else:
    print("No data to export.")

Markdown Exported Successfully!


## 9. Summary

We print out a high-level summary of the entire data dictionary, showing the total number of datasets and total number of columns processed.

In [9]:
print("="*60)
print("DATA DICTIONARY SUMMARY")
print("="*60)

if not dictionary_df.empty:
    print("Datasets:", dictionary_df["Dataset"].nunique())
    print("Total Columns:", len(dictionary_df))
    print("\nPreview:")
    display(dictionary_df.head(20))
else:
    print("No data available to summarize.")

DATA DICTIONARY SUMMARY
Datasets: 6
Total Columns: 48

Preview:


,Dataset,Column Name,Data Type,Missing Values,Missing %,Unique Values,Sample Value
0,analysis.xlsx,Bluestock Fintech — Nifty 100 | Analysis | ...,str,0,0.00,21,id
1,analysis.xlsx,Unnamed: 1,str,0,0.00,6,company_id
2,analysis.xlsx,Unnamed: 2,str,0,0.00,21,compounded_sales_growth
3,analysis.xlsx,Unnamed: 3,str,0,0.00,19,compounded_profit_growth
4,analysis.xlsx,Unnamed: 4,str,0,0.00,21,stock_price_cagr
5,analysis.xlsx,Unnamed: 5,str,0,0.00,21,roe
6,cashflow.xlsx,Bluestock Fintech — Nifty 100 | Cash Flow |...,object,0,0.00,1188,id
7,cashflow.xlsx,Unnamed: 1,str,0,0.00,101,company_id
8,cashflow.xlsx,Unnamed: 2,str,0,0.00,52,year
9,cashflow.xlsx,Unnamed: 3,object,2,0.17,1107,operating_activity
